# Dinámica inmobiliaria en el Eje Cafetero

## Propósito del análisis

Este notebook presenta la primera etapa técnica del proyecto definido en las Entregas 2 y 3: preparar y comprender los registros de compraventas que pueden servir como referencia para el **Observatorio Inmobiliario Catastral (OIC) del IGAC**. El objetivo posterior es estimar un valor esperado para una compraventa y priorizar transacciones que se aparten de los patrones observados, para que un profesional las revise.


## Alcance definido

| Elemento | Decisión del proyecto |
|---|---|
| Territorio inicial | Caldas, Risaralda y Quindío |
| Periodo | 2015–2023, como línea base histórica |
| Operación | `COMPRAVENTA`, `COD_NATUJUR = 125` |
| Fuente principal | Registro de Transacciones Inmobiliarias del IGAC, datos.gov.co |
| Universo consultado | 457.558 registros dentro de este alcance |
| Unidad de análisis preliminar | Un acto registral identificado por `pk` |

La transferencia posterior a Medellín, Bogotá, Cali y Barranquilla es secundaria y solo se evaluará si existe cobertura y calidad suficientes.

## Cómo leer este notebook

Primero se cargan los datos y se aplican reglas explícitas de calidad. Después se revisan faltantes, duplicados, tipos, distribuciones, relaciones y diferencias territoriales.


## Conceptos clave para interpretar los datos

- **Registro administrativo:** cada fila representa un acto registrado, no necesariamente un inmueble único ni una operación económica completa.
- **`pk`:** identifica un registro específico dentro de la fuente. Se usa para revisar duplicados.
- **`matricula`:** folio de matrícula inmobiliaria. Es la variable que se explorará para reconocer posibles ventas repetidas del mismo folio; no se debe asumir automáticamente que equivale a un inmueble inmutable.
- **`valor`:** valor registrado en la fuente. Es la variable monetaria principal, pero puede faltar, ser no válido, ser simbólico o reflejar más de un inmueble.
- **`tiene_valor`:** indicador de disponibilidad del valor y debe contrastarse con `valor`.
- **`folios_derivados`, `count_a`, `count_de` y `tiene_mas_de_un_valor`:** campos que ayudan a identificar divisiones, agrupaciones o complejidades del folio antes de construir ventas repetidas.

Por esto, un `pk` único no significa que el inmueble sea único: una misma `matricula` puede aparecer en distintos años y actos.

## Riesgos que controlamos desde la preparación

La RBS y el AMEF de la Entrega 3 identifican riesgos que afectan directamente esta etapa:

- **Identificación:** una matrícula inconsistente o folios derivados puede impedir reconocer correctamente ventas repetidas.
- **Calidad:** valores faltantes, inválidos, simbólicos, duplicados o escrituras que agrupan varios inmuebles pueden distorsionar las referencias.
- **Temporalidad:** usar información futura para explicar una transacción pasada produciría fuga de información; las validaciones posteriores deben respetar el orden temporal.
- **Capacidad técnica:** la API y la memoria pueden limitar la descarga; por eso el proceso cuenta el universo, pagina la extracción y conserva resultados intermedios.
- **Uso de resultados:** las alertas deben mostrar un motivo y comunicarse como apoyo a la revisión, nunca como una conclusión automática.

Cada una de estas preocupaciones se traduce en una comprobación, una variable de calidad o una limitación documentada en las celdas siguientes.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.igac_inmobiliario.api import SocrataClient
from src.igac_inmobiliario.cleaning import clean_transactions, profile_transactions
from src.igac_inmobiliario.scope import ANALYSIS_COLUMNS, TARGET_DEPARTMENTS, build_scope_where
pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid")

## 1. Carga y limpieza de la base de datos

La consulta es paginada y se limita al universo analítico del proyecto: compraventas de Caldas, Risaralda y Quindío entre 2015 y 2023. No se descarga la base nacional completa; el filtro se aplica en Socrata antes de transferir los datos. `FULL_DATASET = True` procesa todo el universo filtrado y puede cambiarse a `False` para una muestra balanceada de desarrollo.

In [ ]:
client = SocrataClient()
metadata = client.metadata()
print(metadata["name"])
print("Departamentos:", ", ".join(TARGET_DEPARTMENTS))
print("Filtro:", build_scope_where())
print("Variables consultadas:", len(ANALYSIS_COLUMNS.split(",")))

In [ ]:
FULL_DATASET = True
SAMPLE_SIZE = 2_000  # usado solo si FULL_DATASET = False
COMPLETE_DATASET_PATH = ROOT / "data" / "processed" / "compraventas_eje_cafetero_completo_limpio.csv"
where = build_scope_where()

if FULL_DATASET and COMPLETE_DATASET_PATH.exists():
    # Reutiliza la extracción completa ya validada para no repetir una descarga de 108 MB.
    raw = pd.read_csv(COMPLETE_DATASET_PATH)
    expected_rows = len(raw)
    print(f"Extracción completa cargada desde: {COMPLETE_DATASET_PATH}")
else:
    expected_rows = client.count(where=where)
    print(f"Filas esperadas en el universo completo: {expected_rows:,}")
    if FULL_DATASET:
        raw = client.fetch_all(
            where=where,
            select=ANALYSIS_COLUMNS,
            page_size=5_000,
            expected_rows=expected_rows,
        )
    else:
        muestras_por_departamento = []
        for departamento in TARGET_DEPARTMENTS:
            where_departamento = (
                f"departamento = '{departamento}' "
                "and year_radica between '2015' and '2023' "
                "and cod_natujur = '125'"
            )
            muestras_por_departamento.append(client.fetch(
                limit=SAMPLE_SIZE, page_size=1_000,
                where=where_departamento, select=ANALYSIS_COLUMNS,
            ))
        raw = pd.concat(muestras_por_departamento, ignore_index=True)

print(f"Filas descargadas: {len(raw):,}")
print(f"Columnas descargadas: {len(raw.columns):,}")
raw.head(3)


In [ ]:
transactions = clean_transactions(raw)
profile = profile_transactions(transactions)
profile

### Exploración inicial del data frame

La función reproduce el cuadro de revisión estructural del notebook de referencia y agrega duplicados y tasas de faltantes.

In [ ]:
def explorar_dataframe(df):
    return pd.DataFrame({
        "Tipo de dato": df.dtypes,
        "Valores únicos": df.nunique(dropna=True),
        "Datos válidos": df.notna().sum(),
        "Datos faltantes": df.isna().sum(),
        "% faltantes": (df.isna().mean() * 100).round(2),
    }).sort_values("% faltantes", ascending=False)

estructura = explorar_dataframe(transactions)
estructura

In [ ]:
reglas_calidad = pd.Series({
    "Registros descargados": len(transactions),
    "PK duplicados": transactions["pk"].duplicated().sum(),
    "Matrículas faltantes": transactions["matricula"].isna().sum(),
    "TIENE_VALOR = 1": transactions["tiene_valor"].eq(1).sum(),
    "VALOR positivo": transactions["valor"].gt(0).sum(),
    "Valores válidos para universo modelado": transactions["incluido_universo_modelado"].sum(),
})
reglas_calidad.to_frame("conteo")

## 1.1 Análisis preliminar de variables

Se presentan estadísticas descriptivas, histogramas y un correlograma numérico. `VALOR` se examina en escala logarítmica para reducir la dominancia visual de valores extremos; no se modifica el valor original.

In [ ]:
numeric_columns = transactions.select_dtypes(include=np.number).columns.tolist()
categorical_columns = transactions.select_dtypes(exclude=np.number).columns.tolist()

print("Variables numéricas:")
display(transactions[numeric_columns].describe().T)
print("Variables categóricas:")
display(transactions[categorical_columns].describe().T)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
valor_positivo = transactions.loc[transactions["valor"] > 0, "valor"].dropna()
axes[0].hist(valor_positivo, bins=30, color="skyblue", edgecolor="white")
axes[0].set_title("Distribución de VALOR positivo")
axes[0].set_xlabel("Pesos corrientes")
axes[0].set_ylabel("Frecuencia")
axes[0].set_xscale("log")
axes[1].hist(np.log1p(valor_positivo), bins=30, color="salmon", edgecolor="white")
axes[1].set_title("Distribución de log(1 + VALOR)")
axes[1].set_xlabel("Escala logarítmica")
axes[1].set_ylabel("Frecuencia")
year_counts = transactions["year_radica"].value_counts().sort_index()
axes[2].plot(year_counts.index.astype(str), year_counts.values, marker="o", color="seagreen")
axes[2].set_title("Compraventas por año")
axes[2].set_xlabel("Año de radicación")
axes[2].set_ylabel("Registros")
axes[2].tick_params(axis="x", rotation=45)
plt.tight_layout()

In [ ]:
corr_columns = [column for column in numeric_columns if not column.startswith("tiene_")]
plt.figure(figsize=(12, 9))
sns.heatmap(transactions[corr_columns].corr(numeric_only=True), annot=False, cmap="coolwarm", center=0)
plt.title("Correlograma de variables numéricas")
plt.tight_layout()

In [ ]:
by_department = (transactions.groupby(["departamento", "year_radica"], dropna=False)
                 .agg(registros=("pk", "size"), valor_mediano=("valor", "median"), valor_total=("valor", "sum"),
                      matriculas=("matricula", "nunique"), valores_validos=("incluido_universo_modelado", "sum"))
                 .reset_index())
by_department.head(15)

In [ ]:
# Verificación de cobertura territorial
departamentos_esperados = {"CALDAS", "RISARALDA", "QUINDÍO"}

conteo_departamentos = (
    transactions["departamento"]
    .value_counts()
    .reindex(sorted(departamentos_esperados), fill_value=0)
    .rename_axis("departamento")
    .reset_index(name="registros")
)

display(conteo_departamentos)

departamentos_observados = set(
    transactions["departamento"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
    .unique()
)

faltantes = departamentos_esperados - departamentos_observados
adicionales = departamentos_observados - departamentos_esperados

print("Departamentos esperados:", sorted(departamentos_esperados))
print("Departamentos encontrados:", sorted(departamentos_observados))

if faltantes:
    raise ValueError(f"No se encontraron estos departamentos: {sorted(faltantes)}")

if adicionales:
    raise ValueError(f"Se encontraron departamentos fuera del alcance: {sorted(adicionales)}")

print("Verificación correcta: la extracción contiene Caldas, Risaralda y Quindío.")

## 1.2 Tablas y gráficos para comparar departamentos

Las siguientes salidas combinan tablas de contingencia, tablas resumen y gráficos complementarios. Se reportan conteos y medianas, evitando usar el promedio de `VALOR` como único resumen debido a la asimetría y a la presencia de valores extremos.

In [ ]:
# Tabla de contingencia: registros por departamento y año
tabla_contingencia = pd.crosstab(
    transactions["departamento"],
    transactions["year_radica"],
    margins=True,
    margins_name="Total",
)
display(tabla_contingencia)

# Tabla resumen: volumen, cobertura y medidas robustas del valor
tabla_resumen = (
    transactions.groupby("departamento", dropna=False)
    .agg(
        registros=("pk", "size"),
        municipios=("municipio", "nunique"),
        matriculas=("matricula", "nunique"),
        valores_validos=("incluido_universo_modelado", "sum"),
        valor_mediano=("valor", "median"),
        valor_p25=("valor", lambda serie: serie.quantile(0.25)),
        valor_p75=("valor", lambda serie: serie.quantile(0.75)),
    )
    .assign(porcentaje_valores_validos=lambda tabla: tabla["valores_validos"] / tabla["registros"] * 100)
    .sort_values("registros", ascending=False)
)
display(tabla_resumen.style.format({"porcentaje_valores_validos": "{:.1f}%", "valor_mediano": "{:,.0f}", "valor_p25": "{:,.0f}", "valor_p75": "{:,.0f}"}))

### 1.3 Visualización exploratoria con tablas de contingencia, tablas resumen, histogramas, series temporales, barras agrupadas, diagramas de caja y dispersión. Los valores monetarios se muestran también en escala logarítmica por su asimetría; las cajas permiten revisar medianas, rango intercuartílico y posibles atípicos.


### 1.4 Agregaciones exploratorias por año, departamento y municipio.


### 1.5 Analisis estadísticos antes del modelamiento 

### 1.6 Exportación de resultados reproducibles a data/processed/.